```
# Program: DuET v1.1.0
# Author: Sungho Lee, Jae-Won Lee
# Affiliation: MOGAM Institute for Biomedical Research
# Contact: https://github.com/mogam-ai/DuET/issues
# Citation: TBD
```

# Sequence Feature XGBoost Baseline

Train XGBoost on sequence features to establish a non-deep-learning baseline for
TE prediction, using the same 10-fold CV split as DuET for a fair comparison.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
from xgboost import XGBRegressor


## 1. Load Data

In [ ]:
# Paths (same as DuET v5 training)
te_path = "datasets/celltype_te/all-celltype_TE.tsv"
feature_path = "datasets/sequence_features.tsv"

# Label column: "logratio_te" or "residual_te"
label_column = "logratio_te"

# Load TE labels (already log-ratio transformed, no additional log1p needed)
te_data = pd.read_csv(te_path, sep="\t", usecols=["txID", label_column])
te_data.rename(columns={label_column: "label"}, inplace=True)

# Load sequence features
feat_data = pd.read_csv(feature_path, sep="\t")

# Metadata columns to drop (non-numeric identifiers)
meta_cols = ["txID", "geneID", "txName", "geneName", "chrom", "start", "end", "strand"]
feature_cols = [c for c in feat_data.columns if c not in meta_cols]

# Merge
data = te_data.merge(feat_data[["txID"] + feature_cols], on="txID", how="inner")
print(f"Merged data: {data.shape[0]} transcripts, {len(feature_cols)} features")
print(f"Label: {label_column} (no additional transform)")


In [ ]:
# Identify and remove uninformative/redundant features for the main XGBoost run
# (Per-group analysis in Section 5 uses its own feature sets, unaffected by this filtering)

# 1. Constant/near-constant features (variance ~ 0)
variances = data[feature_cols].var()
low_var_cols = variances[variances < 1e-6].index.tolist()
print(f"Near-constant features ({len(low_var_cols)}): {low_var_cols}")

# 2. Remove 3'UTR and full-sequence features (DuET uses only 5'UTR + CDS)
utr3_fullseq_cols = [c for c in feature_cols if "utr3" in c.lower() or "fullseq" in c.lower()]
print(f"\n3'UTR/fullseq cols to drop ({len(utr3_fullseq_cols)}): {utr3_fullseq_cols[:10]}...")

# 2b. Remove cds_aaFreq* (stop codon frequency = 1/cdsLen, r=1.0)
stop_codon_cols = [c for c in feature_cols if c == "cds_aaFreq*"]
print(f"Stop codon freq col to drop: {stop_codon_cols}")

# 2. Prefer logLength over raw length (high correlation, log is more informative)
length_raw_cols = [c for c in feature_cols if "Len" in c and "log" not in c.lower()]
length_log_cols = [c for c in feature_cols if "logLength" in c]
print(f"\nRaw length cols to drop (keep logLength): {length_raw_cols}")

# 3. Drop aaFreq columns that are highly correlated (|r|>0.95) with a codonFreq column
import scipy.stats as stats
aa_freq_cols = [c for c in feature_cols if "aaFreq" in c]
codon_freq_cols = [c for c in feature_cols if "codonFreq" in c]
aa_to_drop = []
for aa_col in aa_freq_cols:
    for cd_col in codon_freq_cols:
        r, _ = stats.pearsonr(data[aa_col].fillna(0), data[cd_col].fillna(0))
        if abs(r) > 0.95:
            aa_to_drop.append((aa_col, cd_col, r))
            break
aa_freq_cols = [x[0] for x in aa_to_drop]
print(f"aaFreq cols to drop (corr>0.95 with codonFreq, {len(aa_freq_cols)}):")
for aa_col, cd_col, r in aa_to_drop:
    print(f"  {aa_col} ~ {cd_col} (r={r:.3f})")

# 4. Features highly correlated with length (|r|>0.95, excluding the length cols themselves)
keep_length_cols = set(length_log_cols)
redundant_with_length = []
for col in feature_cols:
    if col in keep_length_cols or col in length_raw_cols or col in aa_freq_cols or col in low_var_cols:
        continue
    for lcol in length_log_cols:
        r, _ = stats.pearsonr(data[col].fillna(0), data[lcol].fillna(0))
        if abs(r) > 0.95:
            redundant_with_length.append((col, lcol, r))
            break

print(f"\nOther features redundant with logLength (|r|>0.95, {len(redundant_with_length)}):")
for col, lcol, r in redundant_with_length:
    print(f"  {col} ~ {lcol} (r={r:.3f})")

# 5. Highly correlated non-length pairs (|r|>0.98)
remaining = [c for c in feature_cols if c not in set(low_var_cols + length_raw_cols + aa_freq_cols)
             and c not in [x[0] for x in redundant_with_length]]
corr_matrix = data[remaining].fillna(0).corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(col, row, upper_tri.loc[row, col])
                   for col in upper_tri.columns for row in upper_tri.index
                   if upper_tri.loc[row, col] > 0.98]
high_corr_pairs.sort(key=lambda x: -x[2])
drop_from_pairs = set()
for c1, c2, r in high_corr_pairs:
    if c1 not in drop_from_pairs:
        drop_from_pairs.add(c2)

print(f"\nHighly correlated pairs (|r|>0.98, dropping {len(drop_from_pairs)}):")
for c1, c2, r in high_corr_pairs[:15]:
    print(f"  {c1} ~ {c2} (r={r:.3f}) -> drop {c2 if c2 in drop_from_pairs else c1}")

# Build filtered feature set for main XGBoost (Section 5 per-group is unaffected)
cols_to_drop = set(low_var_cols + utr3_fullseq_cols + stop_codon_cols + length_raw_cols + aa_freq_cols)
cols_to_drop.update(col for col, _, _ in redundant_with_length)
cols_to_drop.update(drop_from_pairs)

all_feature_cols = feature_cols  # preserve original for per-group analysis
feature_cols = [c for c in feature_cols if c not in cols_to_drop]
print(f"\nDropped: {len(cols_to_drop)} | Remaining: {len(feature_cols)} features")


In [ ]:
# TE vs sequence length correlations
from scipy.stats import pearsonr, spearmanr

te_values = data["label"].values
length_features = {
    "utr5Len": data.get("utr5Len", data.get("seqLen", pd.Series())),
    "cdsLen": data.get("cdsLen", pd.Series()),
    "utr3Len": data.get("utr3Len", pd.Series()),
}
# Also check logLength versions
for col in all_feature_cols:
    if "logLength" in col:
        length_features[col] = data[col]

print("TE vs length correlations:")
print(f"{'Feature':<25} {'Pearson':>10} {'Spearman':>10}")
print("-" * 47)
for name, values in length_features.items():
    if values.empty or values.isna().all():
        continue
    v = values.fillna(0).values
    pcc = pearsonr(te_values, v)[0]
    scc = spearmanr(te_values, v)[0]
    print(f"{name:<25} {pcc:>10.4f} {scc:>10.4f}")


## 2. 10-Fold Cross-Validation (matched to DuET)

In [ ]:
# Same KFold params as DuET v3
kfold_k = 10
kfold_seed = 42

kf = KFold(n_splits=kfold_k, shuffle=True, random_state=kfold_seed)

X = data[feature_cols].fillna(0).values
y = data["label"].values

results = []
predictions = np.zeros(len(y))

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=60,
        verbosity=0,
    )
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    y_pred = model.predict(X_test)
    predictions[test_idx] = y_pred

    pcc = pearsonr(y_test, y_pred)[0]
    scc = spearmanr(y_test, y_pred)[0]
    r2 = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({"fold": fold, "pearson": pcc, "spearman": scc, "r2": r2, "mae": mae, "rmse": rmse})
    print(f"Fold {fold}: PCC={pcc:.4f} SCC={scc:.4f} R2={r2:.4f} MAE={mae:.4f} RMSE={rmse:.4f}")

results_df = pd.DataFrame(results)
print("Mean: PCC={:.4f} SCC={:.4f} R2={:.4f} MAE={:.4f} RMSE={:.4f}".format(results_df.pearson.mean(), results_df.spearman.mean(), results_df.r2.mean(), results_df.mae.mean(), results_df.rmse.mean()))


## 3. Feature Importance

In [ ]:
# Train final model on all data for feature importance
final_model = XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=60, verbosity=0,
)
final_model.fit(X, y)

# Compute per-feature correlation with TE
from scipy.stats import pearsonr, spearmanr
feat_pcc = [pearsonr(data[col].fillna(0), data["label"])[0] for col in feature_cols]
feat_scc = [spearmanr(data[col].fillna(0), data["label"])[0] for col in feature_cols]

fi = pd.DataFrame({"feature": feature_cols, "importance": final_model.feature_importances_,
                   "pcc_with_te": feat_pcc, "scc_with_te": feat_scc})
fi = fi.sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 20 features:")
print(fi.head(20).to_string())
fi.to_csv("xgb_feature_importance.tsv", sep="\t", index=False)


In [ ]:
# Plot top 20
fig, ax = plt.subplots(figsize=(10, 7))
top_fi = fi.head(20)
sns.barplot(x="importance", y="feature", data=top_fi, palette="viridis", ax=ax)
ax.set_title("XGBoost Feature Importance (Top 20)")
ax.set_xlabel("Importance (gain)")
plt.tight_layout()
plt.savefig("_logs/xgb_feature_importance.png", dpi=150)
plt.show()


## 4. Scatter Plot

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(predictions, y, s=5, alpha=0.5, color="black")
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--", lw=1)
plt.xlabel("Predicted TE")
plt.ylabel("Actual TE")
plt.title(f"XGBoost 10-fold CV (SCC={results_df.spearman.mean():.4f})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("_logs/xgb_scatter.png", dpi=150)
plt.show()


## 5. Per-Feature-Group Analysis

In [ ]:
# Evaluate XGBoost using only features from each DuET feature group
feature_groups = {
    "log_length": [c for c in all_feature_cols if "logLength" in c or "Len" in c.lower()],
    "number_of_aug": [c for c in all_feature_cols if "numAUG" in c or "numORF" in c],
    "base_frequency": [c for c in all_feature_cols if "baseFreq" in c],
    "normalized_mfe": [c for c in all_feature_cols if "normMFE" in c or "mfe" in c.lower()],
    "tis_context": [c for c in all_feature_cols if "startcontext" in c or "kozak" in c.lower()],
    "codon_frequency": [c for c in all_feature_cols if "codonFreq" in c],
    "amino_acid_frequency": [c for c in all_feature_cols if "aaFreq" in c],
}

group_results = []
for group_name, cols in feature_groups.items():
    if not cols:
        print(f"{group_name}: no matching columns")
        continue
    X_group = data[cols].fillna(0).values
    sccs = []
    for train_idx, test_idx in kf.split(X_group):
        m = XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                         random_state=42, n_jobs=60, verbosity=0)
        m.fit(X_group[train_idx], y[train_idx])
        scc = spearmanr(y[test_idx], m.predict(X_group[test_idx]))[0]
        sccs.append(scc)
    mean_scc = np.mean(sccs)
    group_results.append({"group": group_name, "n_features": len(cols), "mean_spearman": mean_scc})
    print(f"{group_name} ({len(cols)} features): SCC={mean_scc:.4f}")

group_results_df = pd.DataFrame(group_results)
print(f"All features combined: SCC={results_df.spearman.mean():.4f}")
